In [1]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Load Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train classifier
model = LogisticRegression()
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



In [9]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
import numpy as np

# Load Iris dataset
iris = load_iris()
X = iris.data
y = iris.target.reshape(-1, 1)  # shape (150,1)

# One-hot encode target for softmax output
encoder = OneHotEncoder(sparse_output=False)
Y = encoder.fit_transform(y)  # shape (150,3)

# Train-test split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42)


In [10]:
def binary_step(x):
    # Binary perceptron activation (0 or 1)
    return np.where(x >= 0, 1, 0)

def bipolar_step(x):
    # Bipolar perceptron activation (-1 or 1)
    return np.where(x >= 0, 1, -1)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))  # for numerical stability
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def cross_entropy(predictions, targets):
    # predictions and targets are both (n_samples, n_classes)
    return -np.sum(targets * np.log(predictions + 1e-9)) / predictions.shape[0]

def accuracy(predictions, targets):
    pred_labels = np.argmax(predictions, axis=1)
    true_labels = np.argmax(targets, axis=1)
    return np.mean(pred_labels == true_labels)


In [11]:
class MLP:
    def __init__(self, input_dim, hidden_dim, output_dim, hidden_activation='binary'):
        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.1
        self.b1 = np.zeros((1, hidden_dim))
        self.W2 = np.random.randn(hidden_dim, output_dim) * 0.1
        self.b2 = np.zeros((1, output_dim))
        
        if hidden_activation == 'binary':
            self.hidden_activation = binary_step
        elif hidden_activation == 'bipolar':
            self.hidden_activation = bipolar_step
        else:
            raise ValueError("hidden_activation must be 'binary' or 'bipolar'")
    
    def forward(self, X):
        self.z1 = X @ self.W1 + self.b1  # linear transform
        self.a1 = self.hidden_activation(self.z1)  # hidden activation
        
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = softmax(self.z2)  # output softmax
        
        return self.a2
    
    def backward(self, X, Y, output, learning_rate=0.1):
        # Output layer error
        delta2 = output - Y  # cross entropy derivative with softmax output
        
        # Gradients for W2 and b2
        dW2 = self.a1.T @ delta2
        db2 = np.sum(delta2, axis=0, keepdims=True)
        
        # Backprop to hidden layer
        delta1 = delta2 @ self.W2.T
        
        # Derivative of hidden activation for perceptron step functions:
        # Since step function derivative is zero everywhere except at 0,
        # we approximate derivative as 1 for simplicity in this educational example.
        # A true perceptron update doesn't use gradient descent like this.
        # This is a simplification to enable training.
        
        if self.hidden_activation == binary_step:
            d_hidden = np.ones_like(self.a1)  # assume constant gradient
        else:  # bipolar_step
            d_hidden = np.ones_like(self.a1)
        
        delta1 *= d_hidden
        
        dW1 = X.T @ delta1
        db1 = np.sum(delta1, axis=0, keepdims=True)
        
        # Gradient descent update
        self.W2 -= learning_rate * dW2
        self.b2 -= learning_rate * db2
        self.W1 -= learning_rate * dW1
        self.b1 -= learning_rate * db1
    
    def train(self, X, Y, epochs=100, lr=0.1):
        for epoch in range(epochs):
            output = self.forward(X)
            loss = cross_entropy(output, Y)
            self.backward(X, Y, output, learning_rate=lr)
            if epoch % 10 == 0 or epoch == epochs - 1:
                acc = accuracy(output, Y)
                print(f"Epoch {epoch}: Loss={loss:.4f}, Accuracy={acc:.4f}")


In [12]:
print("Training MLP with Binary Hidden Activation:")
mlp_binary = MLP(input_dim=4, hidden_dim=5, output_dim=3, hidden_activation='binary')
mlp_binary.train(X_train, Y_train, epochs=100, lr=0.1)

print("\nTraining MLP with Bipolar Hidden Activation:")
mlp_bipolar = MLP(input_dim=4, hidden_dim=5, output_dim=3, hidden_activation='bipolar')
mlp_bipolar.train(X_train, Y_train, epochs=100, lr=0.1)

# Evaluate on test set
output_binary_test = mlp_binary.forward(X_test)
output_bipolar_test = mlp_bipolar.forward(X_test)

print(f"\nTest Accuracy Binary Hidden: {accuracy(output_binary_test, Y_test):.4f}")
print(f"Test Accuracy Bipolar Hidden: {accuracy(output_bipolar_test, Y_test):.4f}")


Training MLP with Binary Hidden Activation:
Epoch 0: Loss=1.1036, Accuracy=0.3524
Epoch 10: Loss=2.1709, Accuracy=0.2952
Epoch 20: Loss=3.5960, Accuracy=0.3524
Epoch 30: Loss=2.0041, Accuracy=0.2952
Epoch 40: Loss=2.8695, Accuracy=0.3524
Epoch 50: Loss=1.5915, Accuracy=0.4762
Epoch 60: Loss=2.8035, Accuracy=0.3048
Epoch 70: Loss=0.8550, Accuracy=0.3524
Epoch 80: Loss=1.1612, Accuracy=0.6095
Epoch 90: Loss=1.2006, Accuracy=0.6095
Epoch 99: Loss=1.3993, Accuracy=0.6476

Training MLP with Bipolar Hidden Activation:
Epoch 0: Loss=1.0969, Accuracy=0.3524
Epoch 10: Loss=14.6005, Accuracy=0.2952
Epoch 20: Loss=7.8402, Accuracy=0.3524
Epoch 30: Loss=7.7986, Accuracy=0.3524
Epoch 40: Loss=7.1787, Accuracy=0.6476
Epoch 50: Loss=13.0580, Accuracy=0.3524
Epoch 60: Loss=8.4701, Accuracy=0.3524
Epoch 70: Loss=7.3025, Accuracy=0.6476
Epoch 80: Loss=9.5110, Accuracy=0.3714
Epoch 90: Loss=7.4345, Accuracy=0.6286
Epoch 99: Loss=5.4056, Accuracy=0.6476

Test Accuracy Binary Hidden: 0.2889
Test Accuracy B

In [13]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load Iris dataset
iris = load_iris()
X = iris.data  # features
y = iris.target  # multiclass labels 0,1,2

# Convert to binary classification: Classify 'Iris-setosa' (0) vs others (1)
y_binary = (y != 0).astype(int)  # 0 if setosa, 1 if not setosa

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_binary, test_size=0.2, random_state=42)

# Activation functions as before
def step_binary(x):
    return 1 if x >= 0 else 0

def step_bipolar(x):
    return 1 if x >= 0 else -1

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def bipolar_sigmoid(x):
    return (2 / (1 + np.exp(-x))) - 1

class Perceptron:
    def __init__(self, learning_rate=0.01, epochs=1000, activation="step_binary"):
        self.lr = learning_rate
        self.epochs = epochs
        self.activation = activation

    def fit(self, X, y):
        self.weights = np.zeros(X.shape[1])
        self.bias = 0
        for _ in range(self.epochs):
            for xi, target in zip(X, y):
                net = self.net_input(xi)
                pred = self.activate(net)
                update = self.lr * (target - pred)
                self.weights += update * xi
                self.bias += update

    def net_input(self, X):
        return np.dot(X, self.weights) + self.bias

    def activate(self, x):
        if self.activation == "step_binary":
            return step_binary(x)
        elif self.activation == "step_bipolar":
            return step_bipolar(x)
        elif self.activation == "sigmoid":
            return 1 if sigmoid(x) >= 0.5 else 0
        elif self.activation == "bipolar_sigmoid":
            return 1 if bipolar_sigmoid(x) >= 0 else -1
        else:
            raise ValueError("Invalid activation function")

    def predict(self, X):
        if len(X.shape) == 1:
            return self.activate(self.net_input(X))
        return np.array([self.activate(self.net_input(xi)) for xi in X])

for act in ["step_binary", "step_bipolar", "sigmoid", "bipolar_sigmoid"]:
    perceptron = Perceptron(learning_rate=0.01, epochs=1000, activation=act)
    perceptron.fit(X_train, y_train)
    y_pred = perceptron.predict(X_test)

    # Adjust labels for bipolar activations (use -1 instead of 0)
    if act in ["step_bipolar", "bipolar_sigmoid"]:
        y_test_adj = np.where(y_test == 0, -1, 1)
    else:
        y_test_adj = y_test

    acc = accuracy_score(y_test_adj, y_pred)
    prec = precision_score(y_test_adj, y_pred, average='binary', pos_label=1)
    rec = recall_score(y_test_adj, y_pred, average='binary', pos_label=1)
    f1 = f1_score(y_test_adj, y_pred, average='binary', pos_label=1)

    print(f"\nActivation: {act}")
    print("Final Weights:", np.round(perceptron.weights, 2))
    print("Final Bias:", round(perceptron.bias, 2))
    print("Accuracy :", round(acc * 100, 2), "%")
    print("Precision:", round(prec * 100, 2), "%")
    print("Recall   :", round(rec * 100, 2), "%")
    print("F1 Score :", round(f1 * 100, 2), "%")


Activation: step_binary
Final Weights: [ 0.01 -0.02  0.02  0.02]
Final Bias: 0.01
Accuracy : 100.0 %
Precision: 100.0 %
Recall   : 100.0 %
F1 Score : 100.0 %

Activation: step_bipolar
Final Weights: [0.01 0.02 0.04 0.01]
Final Bias: 0.08
Accuracy : 70.0 %
Precision: 68.97 %
Recall   : 100.0 %
F1 Score : 81.63 %

Activation: sigmoid
Final Weights: [ 0.01 -0.02  0.02  0.02]
Final Bias: 0.01
Accuracy : 100.0 %
Precision: 100.0 %
Recall   : 100.0 %
F1 Score : 100.0 %

Activation: bipolar_sigmoid
Final Weights: [0.01 0.02 0.04 0.01]
Final Bias: 0.08
Accuracy : 70.0 %
Precision: 68.97 %
Recall   : 100.0 %
F1 Score : 81.63 %
